# Learning Paradigms — Exercises

Companion to the note [Learning Paradigms](Learning%20Paradigms.md).

Practise telling the paradigms apart (supervised, unsupervised, semi-/self-supervised, reinforcement, online, transfer), the discriminative vs. generative and parametric vs. non-parametric distinctions, and implement a small representative of several paradigms from scratch.

**17 exercises** · 🧠 Concept ×5 · ✍️ By hand ×5 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.datasets import load_iris, load_digits, make_moons
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, Perceptron, Ridge, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.semi_supervised import LabelSpreading
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
rng = np.random.default_rng(0)

## Part A · Concepts

### Exercise 1 · Which paradigm?
*🧠 Concept · ★☆☆*

Name the paradigm from the note's table for each scenario.

1. 10 000 X-ray images, 200 of them labelled by a radiologist.
2. Train a model to predict the next word of Wikipedia sentences, then reuse its representations.
3. Group customers by purchase behaviour.
4. A robot learns to walk from a reward for forward velocity.
5. A spam filter that updates after every new e-mail is labelled.
6. Take an ImageNet ResNet and adapt it to 500 photos of plant diseases.
7. Predict a house price from 20 features with 50 000 past sales.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask: where does the training signal come from (human labels, the data itself, a reward, nothing)? And does data arrive all at once or as a stream?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1 semi-supervised · 2 self-supervised (labels = the next word, derived from the data) · 3 unsupervised (clustering) ·
4 reinforcement · 5 online (supervised, incremental) · 6 transfer / fine-tuning · 7 supervised regression.
Paradigms can combine: 2 is self-supervised **pre-training** followed by transfer.

</details>

### Exercise 2 · Self-supervised is not unsupervised
*🧠 Concept · ★☆☆*

Both self-supervised and unsupervised learning use only $x$. Explain the difference, and give the pretext task (the derived $(x', y')$ pair)
for: (a) masked language modeling, (b) image rotation prediction, (c) contrastive learning on augmented image pairs.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Self-supervised learning manufactures a supervised problem from unlabelled data and trains with an ordinary supervised loss.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Unsupervised learning looks for structure directly (clusters, densities, low-dimensional projections). Self-supervised learning
**creates labels from the data** and solves a supervised task whose real purpose is to learn a good representation.
(a) $x'$ = sentence with some tokens masked, $y'$ = the masked tokens. (b) $x'$ = image rotated by $k\cdot90^\circ$, $y' = k$.
(c) $x'$ = a pair of augmented views, $y'$ = "same image or not"; the loss pulls positive pairs together and pushes negatives apart.

</details>

### Exercise 3 · Parametric vs. non-parametric
*🧠 Concept · ★☆☆*

You train (a) linear regression with $d$ features and (b) $k$-NN regression on $N$ samples. How much must be stored after training,
and how expensive is one prediction, as a function of $N$ and $d$? What happens to each when $N$ grows 100×?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A parametric model compresses the data into a fixed-size $\theta$; a non-parametric one keeps (a function of) the data.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Store $d+1$ numbers, prediction $O(d)$, independent of $N$. (b) Store all $N\cdot d$ inputs and $N$ targets; naive prediction costs $O(Nd)$.
With 100× more data the linear model is unchanged in size (but its capacity is also fixed, so it may underfit), whereas $k$-NN
grows 100× in memory and prediction time, but becomes more flexible: its complexity grows with the data.

</details>

### Exercise 4 · Discriminative or generative?
*🧠 Concept · ★★☆*

For each property, say whether it favours a **generative** model $p(x,y)$ or a **discriminative** model $p(y\mid x)$, and why.

(a) You want to sample new realistic inputs $x$. (b) You only care about classification accuracy and have lots of data.
(c) Some feature values are missing at prediction time. (d) Very few training examples. (e) You want to detect inputs unlike anything in training.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A generative model gives you $p(x)=\sum_y p(x,y)$ for free; a discriminative model spends all its capacity on the decision boundary.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Generative**: only it models $p(x\mid y)$, from which you can sample. (b) **Discriminative**: it does not waste effort modelling $p(x)$
and usually has lower asymptotic error (Ng & Jordan, 2002). (c) **Generative**: marginalise the missing feature out of $p(x\mid y)$
(trivial in [[Naive Bayes]]). (d) **Generative** often wins: its stronger assumptions act as regularization and it converges faster.
(e) **Generative**: low $p(x)$ flags out-of-distribution inputs ([[Anomaly Detection]]).

</details>

### Exercise 5 · When does transfer help?
*🧠 Concept · ★★☆*

You fine-tune a pretrained image network on a new task. Discuss which strategy you would choose — (i) freeze everything except a new
output layer, or (ii) fine-tune all layers with a small learning rate — when (a) the new dataset is tiny and similar to the pretraining data,
(b) it is large and quite different (e.g. medical scans), (c) tiny and very different. What is *negative transfer*?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Early layers learn generic features (edges, textures); late layers are task-specific. More trainable parameters need more data.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Freeze and train only a linear head: features already fit, and few data cannot support many parameters.
(b) Fine-tune everything (small LR, or even train longer): enough data to adapt, and the domain differs.
(c) Hardest case: use only early layers' features, freeze most, heavy augmentation / regularization.
**Negative transfer** is when pretraining hurts compared to training from scratch, because the source task is too dissimilar or misleading.

</details>

## Part B · By hand

### Exercise 6 · A generative classifier with Bayes' rule
*✍️ By hand · ★☆☆*

A generative spam model: $p(\text{spam}) = 0.3$. The binary feature $x$ = "contains the word *free*" has
$p(x=1\mid\text{spam}) = 0.6$ and $p(x=1\mid\text{ham}) = 0.05$. Compute $p(\text{spam}\mid x=1)$ and $p(\text{spam}\mid x=0)$.

In [ ]:
p_spam_given_free = None
p_spam_given_nofree = None

_joint = np.array([[0.7*0.95, 0.7*0.05], [0.3*0.4, 0.3*0.6]])   # rows: ham, spam; cols: x=0, x=1
check('p(spam | x=1)', p_spam_given_free, _joint[1, 1] / _joint[:, 1].sum())
check('p(spam | x=0)', p_spam_given_nofree, _joint[1, 0] / _joint[:, 0].sum())

<details>
<summary><b>💡 Hint</b></summary>

$p(y\mid x) = \dfrac{p(x\mid y)p(y)}{\sum_{y'} p(x\mid y')p(y')}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p(\text{spam}\mid x=1) = \dfrac{0.6\cdot0.3}{0.6\cdot0.3 + 0.05\cdot0.7} = \dfrac{0.18}{0.215} \approx 0.837$.
$p(\text{spam}\mid x=0) = \dfrac{0.4\cdot0.3}{0.4\cdot0.3 + 0.95\cdot0.7} = \dfrac{0.12}{0.785} \approx 0.153$.
The generative model learned $p(x\mid y)$ and $p(y)$ and derives $p(y\mid x)$ from them.

```python
p_spam_given_free = 0.18 / 0.215
p_spam_given_nofree = 0.12 / 0.785
```

</details>

### Exercise 7 · Counting parameters
*✍️ By hand · ★★☆*

Binary classification ($K=2$) with $d=10$ continuous features. Count the free parameters of:

(a) logistic regression; (b) Gaussian class-conditionals with a **full covariance per class** plus class prior;
(c) the same but with **one shared** covariance (LDA); (d) Gaussian naive Bayes (diagonal covariance per class).

In [ ]:
n_logreg = None
n_gda_full = None
n_lda = None
n_gnb = None

_d, _K = 10, 2
_cov = _d * (_d + 1) // 2
check('logistic regression', n_logreg, _d + 1)
check('full-covariance GDA', n_gda_full, _K * _d + _K * _cov + (_K - 1))
check('LDA', n_lda, _K * _d + _cov + (_K - 1))
check('Gaussian NB', n_gnb, _K * _d + _K * _d + (_K - 1))

<details>
<summary><b>💡 Hint</b></summary>

A symmetric $d\times d$ covariance has $d(d+1)/2$ free entries. $K$ class priors that sum to 1 have $K-1$ free values.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $d+1 = 11$. (b) means $2\cdot10$, covariances $2\cdot55$, prior $1$: $131$. (c) $20 + 55 + 1 = 76$. (d) $20 + 20 + 1 = 41$.
Generative models must describe how $x$ is distributed, which costs far more parameters than the decision boundary alone;
the full-covariance model grows as $O(d^2)$.

```python
n_logreg = 11
n_gda_full = 131
n_lda = 76
n_gnb = 41
```

</details>

### Exercise 8 · Discounted return
*✍️ By hand · ★☆☆*

In reinforcement learning the agent maximises the return $G_0 = \sum_{t\ge0}\gamma^t r_{t+1}$.
An episode yields rewards $(1, 0, 2, 5)$ and then ends. Compute $G_0$ for $\gamma = 0.9$ and for $\gamma=0$. What does $\gamma$ control?

In [ ]:
G_09 = None
G_0 = None

_r = np.array([1, 0, 2, 5.])
check('return gamma=0.9', G_09, np.sum(0.9 ** np.arange(4) * _r))
check('return gamma=0', G_0, _r[0])

<details>
<summary><b>💡 Hint</b></summary>

$G_0 = r_1 + \gamma r_2 + \gamma^2 r_3 + \gamma^3 r_4$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\gamma=0.9$: $1 + 0 + 0.81\cdot2 + 0.729\cdot5 = 1 + 1.62 + 3.645 = 6.265$. $\gamma = 0$: $G_0 = 1$ (only the immediate reward).
$\gamma$ sets how far-sighted the agent is. Unlike supervised learning there is no label per action, only delayed rewards
([[Q-Learning and Policy Gradients]]).

```python
G_09 = 6.265
G_0 = 1.0
```

</details>

### Exercise 9 · Online learning: perceptron by hand
*✍️ By hand · ★★☆*

The perceptron processes one example at a time: if $y_i(w^\top x_i + b)\le 0$ then $w\leftarrow w + y_ix_i$, $b\leftarrow b+y_i$.
Start at $w=(0,0)$, $b=0$ and process, in order,
$((1,2),+1)$, $((2,-1),-1)$, $((-1,-2),-1)$, $((3,1),+1)$. Give $w$ and $b$ after one pass.

In [ ]:
w_perc = None  # [w1, w2]
b_perc = None

_Xp = np.array([[1,2],[2,-1],[-1,-2],[3,1.]]); _yp = np.array([1,-1,-1,1])
_pc = Perceptron(eta0=1.0, shuffle=False, max_iter=1, tol=None).fit(_Xp, _yp)
check('w', w_perc, _pc.coef_[0]); check('b', b_perc, _pc.intercept_[0])

<details>
<summary><b>💡 Hint</b></summary>

Score of point 2 after the first update: $(1,2)\cdot(2,-1) + 1 = 1$, which has the wrong sign.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Score $0 \Rightarrow$ update: $w=(1,2)$, $b=1$.
2. Score $2-2+1 = 1$, $y=-1$, wrong $\Rightarrow$ $w=(1-2,\,2+1)=(-1,3)$, $b=0$.
3. Score $1-6+0=-5$, $y=-1$, correct, no update.
4. Score $-3+3+0=0 \Rightarrow$ update: $w=(2,4)$, $b=1$.

Final $w=(2,4)$, $b=1$. Each step uses only the current example: memory is $O(d)$ regardless of stream length.

```python
w_perc = [2, 4]
b_perc = 1
```

</details>

### Exercise 10 · Incremental means for bandits
*✍️ By hand · ★★☆*

An online learner estimates an arm's mean reward without storing history.
(a) Show that the sample mean satisfies $Q_n = Q_{n-1} + \frac1n(r_n - Q_{n-1})$.
(b) For rewards $(1, 0, 0, 1, 1)$ and $Q_0=0$, compute $Q_5$ with step $\frac1n$ and with a constant step $\alpha=0.5$.
(c) When is the constant step preferable?

In [ ]:
Q5_mean = None
Q5_const = None

_rw = [1, 0, 0, 1, 1]
_q = 0.0
for _x in _rw:
    _q += 0.5 * (_x - _q)
check('sample-mean step', Q5_mean, np.mean(_rw)); check('constant step', Q5_const, _q)

<details>
<summary><b>💡 Hint</b></summary>

Write $Q_n = \frac1n\big((n-1)Q_{n-1} + r_n\big)$ and rearrange.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $Q_n = \frac1n\sum_{i\le n} r_i = \frac1n\big((n-1)Q_{n-1} + r_n\big) = Q_{n-1} + \frac1n(r_n - Q_{n-1})$.
(b) Step $1/n$: $Q = 1, 0.5, 0.333, 0.5, 0.6$, so $Q_5 = 0.6$. Step $0.5$: $0.5, 0.25, 0.125, 0.5625, 0.78125$.
(c) A constant $\alpha$ weights recent rewards exponentially more, so it tracks **non-stationary** rewards (drift in online settings).

```python
Q5_mean = 0.6
Q5_const = 0.78125
```

</details>

## Part C · Code

### Exercise 11 · Generative: Gaussian naive Bayes from scratch
*💻 Code · ★★☆*

Implement `GNB` with `fit(X, y)` (store priors, per-class means and variances) and `predict_log_joint(X)` returning
$\ln p(y) + \sum_j \ln \mathcal N(x_j\mid\mu_{yj},\sigma^2_{yj})$ for every class, plus `predict`. Compare with `GaussianNB` on iris.

In [ ]:
class GNB:
    def fit(self, X, y):
        # TODO: self.classes, self.prior, self.mu, self.var
        return self
    def predict_log_joint(self, X):
        return None
    def predict(self, X):
        return None

X_ir, y_ir = load_iris(return_X_y=True)
gnb_pred = GNB().fit(X_ir, y_ir).predict(X_ir)

check('matches sklearn GaussianNB', gnb_pred, GaussianNB(var_smoothing=1e-12).fit(X_ir, y_ir).predict(X_ir))

<details>
<summary><b>💡 Hint</b></summary>

Use the **biased** variance (`X[y==k].var(0)`) as sklearn does. Work in log space and take `argmax` over classes.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Because it models $p(x\mid y)$ the model is generative: you could sample new flowers from $\mathcal N(\mu_y, \operatorname{diag}\sigma^2_y)$.
The "naive" assumption is conditional independence of features given the class ([[Naive Bayes]]).

```python
class GNB:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.prior = np.array([np.mean(y == k) for k in self.classes])
        self.mu = np.array([X[y == k].mean(0) for k in self.classes])
        self.var = np.array([X[y == k].var(0) for k in self.classes])
        return self
    def predict_log_joint(self, X):
        ll = -0.5 * (np.log(2 * np.pi * self.var)[None] + (X[:, None, :] - self.mu[None]) ** 2 / self.var[None]).sum(-1)
        return ll + np.log(self.prior)[None]
    def predict(self, X):
        return self.classes[np.argmax(self.predict_log_joint(X), axis=1)]

X_ir, y_ir = load_iris(return_X_y=True)
gnb_pred = GNB().fit(X_ir, y_ir).predict(X_ir)
```

</details>

### Exercise 12 · Generative model: sample new data
*💻 Code · ★★☆*

Use your fitted `GNB` to **generate** 1000 synthetic flowers of class 2 (virginica): store them in `X_fake`, shape `(1000, 4)`.
Then classify them with a *discriminative* `LogisticRegression` trained on the real iris data: what fraction is predicted as class 2?
Why can a discriminative model not do the sampling step?

In [ ]:
gnb = GNB().fit(X_ir, y_ir)
X_fake = None
frac_class2 = None

if X_fake is not None:
    check('shape', np.shape(X_fake), (1000, 4))
    check('per-feature means ~ class-2 means', np.abs(np.mean(X_fake, 0) - X_ir[y_ir == 2].mean(0)).max() < 0.1, True)
    check('mostly classified as virginica', frac_class2 > 0.8, True)
else:
    check('shape', None, (1000, 4))

<details>
<summary><b>💡 Hint</b></summary>

`rng.normal(gnb.mu[2], np.sqrt(gnb.var[2]), size=(1000, 4))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Most synthetic flowers (≈ 93 %) are recognised as virginica; the rest fall into versicolor territory because naive Bayes ignores the
correlation between features and so produces some unrealistic combinations. A discriminative model only knows $p(y\mid x)$: it has
no distribution over $x$ to sample from.

```python
gnb = GNB().fit(X_ir, y_ir)
X_fake = rng.normal(gnb.mu[2], np.sqrt(gnb.var[2]), size=(1000, 4))
clf = LogisticRegression(max_iter=1000).fit(X_ir, y_ir)
frac_class2 = np.mean(clf.predict(X_fake) == 2)
print(frac_class2)
```

</details>

### Exercise 13 · Online learning: a streaming perceptron
*💻 Code · ★★☆*

Implement `perceptron_stream(X, y, epochs)` that visits the samples **one at a time** in order and applies the update rule from the
by-hand exercise. Run it on the linearly separable stream below and store the final training accuracy in `acc_stream`.
Compare your weights with `Perceptron(eta0=1, shuffle=False, tol=None, max_iter=epochs)`.

In [ ]:
X_st = rng.normal(size=(200, 2))
y_st = np.where(X_st @ np.array([2.0, -1.0]) + 0.5 > 0, 1, -1)

def perceptron_stream(X, y, epochs=20):
    # TODO: return (w, b)
    return None

res_stream = perceptron_stream(X_st, y_st)
acc_stream = None

_pref = Perceptron(eta0=1.0, shuffle=False, tol=None, max_iter=20).fit(X_st, y_st)
check('matches sklearn weights', None if res_stream is None else np.r_[res_stream[0], res_stream[1]], np.r_[_pref.coef_[0], _pref.intercept_])
check('separable data -> 100% train accuracy', acc_stream, 1.0)

<details>
<summary><b>💡 Hint</b></summary>

Two nested loops: `for epoch`, `for i in range(N)`. The perceptron convergence theorem guarantees finitely many mistakes on separable data.

</details>

<details>
<summary><b>✅ Solution</b></summary>

On separable data the perceptron makes at most $(R/\gamma)^2$ mistakes ($R$ = data radius, $\gamma$ = margin) and then stops updating.
The same loop works on an infinite stream: that is the online paradigm.

```python
X_st = rng.normal(size=(200, 2))
y_st = np.where(X_st @ np.array([2.0, -1.0]) + 0.5 > 0, 1, -1)

def perceptron_stream(X, y, epochs=20):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epochs):
        for xi, yi in zip(X, y):
            if yi * (w @ xi + b) <= 0:
                w = w + yi * xi
                b = b + yi
    return w, b

res_stream = perceptron_stream(X_st, y_st)
w_s, b_s = res_stream
acc_stream = np.mean(np.sign(X_st @ w_s + b_s) == y_st)
```

</details>

### Exercise 14 · Semi-supervised: label propagation
*💻 Code · ★★★*

`make_moons` with 200 points but only **10 labels**. Implement label propagation:
build $W_{ij} = \exp(-\gamma\lVert x_i-x_j\rVert^2)$ with $\gamma = 20$, row-normalise to $T = D^{-1}W$, start with one-hot labels
$F$ (zeros for unlabelled), and repeat $F\leftarrow TF$, then reset the labelled rows, 2000 times. Store accuracy on all 200 points in
`acc_prop` and compare with a logistic regression trained on the 10 labels only (`acc_sup`).

In [ ]:
X_m, y_m = make_moons(200, noise=0.1, random_state=0)
_r = np.random.default_rng(0)
labeled = np.r_[_r.choice(np.where(y_m == 0)[0], 5, replace=False), _r.choice(np.where(y_m == 1)[0], 5, replace=False)]

def label_propagation(X, labeled, y_lab, gamma=20.0, iters=2000):
    # TODO: return predicted labels for all points
    return None

acc_prop = None
acc_sup = None

_yl = -np.ones(200, int); _yl[labeled] = y_m[labeled]
_ref = LabelSpreading(gamma=20).fit(X_m, _yl).transduction_
check('propagation beats supervised-only', None if acc_prop is None else (acc_prop > acc_sup + 0.1), True)
check('agrees with sklearn LabelSpreading on >95% points', None if acc_prop is None else np.mean(label_propagation(X_m, labeled, y_m[labeled]) == _ref) > 0.95, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Pairwise squared distances: `((X[:, None] - X[None]) ** 2).sum(-1)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Keep `Y0 = F[labeled].copy()` and do `F[labeled] = Y0` after every multiplication.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Propagation reaches ≈ 0.99 accuracy, the 10-label logistic regression only ≈ 0.83. The unlabelled points reveal the two moon-shaped
**manifolds**; labels flow along dense regions (the cluster assumption). This only helps when the class structure matches the data's
geometry.

```python
X_m, y_m = make_moons(200, noise=0.1, random_state=0)
_r = np.random.default_rng(0)
labeled = np.r_[_r.choice(np.where(y_m == 0)[0], 5, replace=False), _r.choice(np.where(y_m == 1)[0], 5, replace=False)]

def label_propagation(X, labeled, y_lab, gamma=20.0, iters=2000):
    W = np.exp(-gamma * ((X[:, None] - X[None]) ** 2).sum(-1))
    T = W / W.sum(1, keepdims=True)
    F = np.zeros((len(X), 2)); F[labeled, y_lab] = 1
    Y0 = F[labeled].copy()
    for _ in range(iters):
        F = T @ F
        F[labeled] = Y0
    return F.argmax(1)

acc_prop = np.mean(label_propagation(X_m, labeled, y_m[labeled]) == y_m)
acc_sup = LogisticRegression().fit(X_m[labeled], y_m[labeled]).score(X_m, y_m)
print(acc_prop, acc_sup)
```

</details>

### Exercise 15 · Self-supervised pretext task
*💻 Code · ★★☆*

Labels can be derived from the data itself. Write `make_pretext(images)` that turns 8×8 digit images into a supervised problem:
input = left half (columns 0–3, flattened to 32 values), target = right half (columns 4–7). Fit `Ridge(alpha=1)` on a train split and
store the test $R^2$ (`multioutput='variance_weighted'`) in `r2_pretext`. No digit labels are used anywhere.

In [ ]:
imgs = load_digits().images   # (1797, 8, 8)

def make_pretext(images):
    # TODO: return (inputs, targets)
    return None

r2_pretext = None

if r2_pretext is not None:
    _in, _tg = make_pretext(imgs)
    check('shapes', (_in.shape, _tg.shape), ((1797, 32), (1797, 32)))
    check('better than predicting the mean (R^2 > 0.2)', r2_pretext > 0.2, True)
else:
    check('R^2 > 0.2', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`images[:, :, :4].reshape(len(images), -1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$R^2\approx0.27$: even a linear model learns something about the structure of digits from unlabelled data. Deep self-supervised models
solve much harder pretext tasks (masked patches, next token) and the **intermediate representation** they learn is then reused for the
real task via transfer.

```python
imgs = load_digits().images

def make_pretext(images):
    n = len(images)
    return images[:, :, :4].reshape(n, -1), images[:, :, 4:].reshape(n, -1)

_inp, _tgt = make_pretext(imgs)
a, b, c, d = train_test_split(_inp, _tgt, random_state=0)
r2_pretext = r2_score(d, Ridge(alpha=1.0).fit(a, c).predict(b), multioutput='variance_weighted')
print(r2_pretext)
```

</details>

### Exercise 16 · Reinforcement / online: ε-greedy bandit
*💻 Code · ★★☆*

Simulate a 3-armed Bernoulli bandit with success probabilities `p_arms = [0.2, 0.5, 0.7]` for 2000 steps. With probability
$\varepsilon=0.1$ pick a random arm, otherwise the arm with the highest estimate $Q$; update $Q$ with the incremental mean.
Store the final estimates in `Q_bandit` and the pull counts in `n_pulls`. Use `rng_b = np.random.default_rng(1)`.

In [ ]:
p_arms = np.array([0.2, 0.5, 0.7])
rng_b = np.random.default_rng(1)
Q_bandit = None
n_pulls = None

if Q_bandit is not None:
    check('best arm identified', int(np.argmax(Q_bandit)), 2)
    check('estimate of best arm close to 0.7', abs(Q_bandit[2] - 0.7) < 0.05, True)
    check('best arm pulled most', int(np.argmax(n_pulls)), 2)
else:
    check('best arm identified', None, 2)

<details>
<summary><b>💡 Hint</b></summary>

Reward: `float(rng_b.random() < p_arms[a])`. Update: `n[a] += 1; Q[a] += (r - Q[a]) / n[a]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The agent never sees the "correct" arm, only rewards for the arms it chose: this is the **exploration–exploitation** trade-off that
separates RL from supervised learning. The best arm gets ≈ 90 % of the pulls; estimates of rarely-pulled arms stay noisy.

```python
p_arms = np.array([0.2, 0.5, 0.7])
rng_b = np.random.default_rng(1)
Q_bandit, n_pulls = np.zeros(3), np.zeros(3)
for t in range(2000):
    a = rng_b.integers(3) if rng_b.random() < 0.1 else int(np.argmax(Q_bandit))
    r = float(rng_b.random() < p_arms[a])
    n_pulls[a] += 1
    Q_bandit[a] += (r - Q_bandit[a]) / n_pulls[a]
print(Q_bandit.round(3), n_pulls)
```

</details>

### Exercise 17 · Non-parametric k-NN regression
*💻 Code · ★★☆*

Implement `knn_regress(X_train, y_train, X_query, k)` (average of the $k$ nearest targets, Euclidean distance).
On $y=\sin(x)+\varepsilon$ with $x\in[0, 2\pi]$, compare its training-free fit with a parametric linear regression.
Store both test MSEs in `mse_knn`, `mse_lin`.

In [ ]:
x_tr = np.sort(rng.uniform(0, 2 * np.pi, 150)); y_tr = np.sin(x_tr) + 0.2 * rng.normal(size=150)
x_te = np.linspace(0, 2 * np.pi, 100); y_te = np.sin(x_te)

def knn_regress(X_train, y_train, X_query, k=5):
    return None

pred_knn = knn_regress(x_tr[:, None], y_tr, x_te[:, None], k=5)
mse_knn = None
mse_lin = None

check('k-NN matches sklearn', pred_knn, KNeighborsRegressor(5).fit(x_tr[:, None], y_tr).predict(x_te[:, None]))
check('non-parametric wins on a sine', None if mse_knn is None else mse_knn < mse_lin / 5, True)

<details>
<summary><b>💡 Hint</b></summary>

Distances `np.abs(X_query - X_train.T)` for 1-D, or the general `((Q[:, None] - X[None]) ** 2).sum(-1)`; then `np.argsort(...)[:, :k]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

k-NN adapts to the sine with no assumptions about its shape (MSE ≈ 0.01), the straight line cannot (MSE ≈ 0.2). The price: k-NN stores
all 150 points and its flexibility grows with $N$, while linear regression keeps 2 numbers.

```python
x_tr = np.sort(rng.uniform(0, 2 * np.pi, 150)); y_tr = np.sin(x_tr) + 0.2 * rng.normal(size=150)
x_te = np.linspace(0, 2 * np.pi, 100); y_te = np.sin(x_te)

def knn_regress(X_train, y_train, X_query, k=5):
    D = ((X_query[:, None, :] - X_train[None]) ** 2).sum(-1)
    idx = np.argsort(D, axis=1, kind='stable')[:, :k]
    return y_train[idx].mean(1)

pred_knn = knn_regress(x_tr[:, None], y_tr, x_te[:, None], k=5)
mse_knn = np.mean((pred_knn - y_te) ** 2)
mse_lin = np.mean((LinearRegression().fit(x_tr[:, None], y_tr).predict(x_te[:, None]) - y_te) ** 2)
print(mse_knn, mse_lin)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Learning Paradigms](Learning%20Paradigms.md).*